# Ritter Problem I: Clean Homotopy Algorithm

### Jupyter Notebook Version

This notebook is formatted as **Markdown + MathJax**, so the equations render directly in JupyterLab, classic Jupyter Notebook, VS Code notebooks, and Google Colab.

<div style="padding: 12px 14px; border-left: 5px solid #2f6f9f; background: #f3f8fb; border-radius: 6px; margin-top: 12px;">
<strong>How to use this notebook:</strong> no code execution is required. If an equation appears as raw text, click the cell and press <kbd>Shift</kbd>+<kbd>Enter</kbd> to render the Markdown cell.
</div>

**Notation convention.** The coupling multiplier is always written as $v(t)$. The local block multipliers are written as $u_j(t)$.


## Table of contents

- [1. Original Problem I](#1-original-problem-i)
  - [Coupling constraints](#coupling-constraints)
  - [Local block constraints](#local-block-constraints)
- [2. Why we do not just put the $B_j$'s into the $A_j$'s](#2-why-we-do-not-just-put-the-bjs-into-the-ajs)
  - [Coupling constraints need shared prices](#coupling-constraints-need-shared-prices)
  - [Local constraints should stay local](#local-constraints-should-stay-local)
  - [What goes wrong if you put $B$'s into the master](#what-goes-wrong-if-you-put-bs-into-the-master)
  - [The correct interpretation](#the-correct-interpretation)
- [3. Relaxed homotopy problem](#3-relaxed-homotopy-problem)
- [4. Starting point](#4-starting-point)
- [5. Active sets](#5-active-sets)
  - [Active coupling set](#active-coupling-set)
  - [Active local block sets](#active-local-block-sets)
- [6. Multipliers](#6-multipliers)
- [7. KKT equations for one fixed active set](#7-kkt-equations-for-one-fixed-active-set)
- [8. Solve each block in terms of $v$ and $t$](#8-solve-each-block-in-terms-of-v-and-t)
- [9. Determine $v$ from active coupling constraints](#9-determine-v-from-active-coupling-constraints)
- [10. Now every unknown is affine in $t$](#10-now-every-unknown-is-affine-in-t)
- [11. What still needs to be checked](#11-what-still-needs-to-be-checked)
- [12. Generic blocker rule](#12-generic-blocker-rule)
- [13. Four blocker types](#13-four-blocker-types)
  - [Blocker 1: inactive local constraint becomes tight](#blocker-1-inactive-local-constraint-becomes-tight)
  - [Blocker 2: active local multiplier becomes zero](#blocker-2-active-local-multiplier-becomes-zero)
  - [Blocker 3: inactive coupling constraint becomes tight](#blocker-3-inactive-coupling-constraint-becomes-tight)
  - [Blocker 4: active coupling multiplier becomes zero](#blocker-4-active-coupling-multiplier-becomes-zero)
- [14. The iteration](#14-the-iteration)
- [15. Why the inactive constraints stay valid](#15-why-the-inactive-constraints-stay-valid)
- [16. Why active multipliers stay valid](#16-why-active-multipliers-stay-valid)
- [17. Why the four cases are exactly enough](#17-why-the-four-cases-are-exactly-enough)
- [18. What happens at a breakpoint](#18-what-happens-at-a-breakpoint)
  - [If a constraint is added](#if-a-constraint-is-added)
  - [If a constraint is dropped](#if-a-constraint-is-dropped)
- [19. Degeneracy and same-$t$ blockers](#19-degeneracy-and-same-t-blockers)
  - [Dropping constraints is safe](#dropping-constraints-is-safe)
  - [Adding constraints needs independence](#adding-constraints-needs-independence)
- [20. Convergence idea](#20-convergence-idea)
- [21. Why reaching $t=0$ proves optimality](#21-why-reaching-t0-proves-optimality)
- [22. Compact pseudocode](#22-compact-pseudocode)
- [23. How to explain the algorithm in words](#23-how-to-explain-the-algorithm-in-words)
- [24. The one-line memory version](#24-the-one-line-memory-version)

## Notebook rendering notes

This file intentionally uses notebook-native math delimiters:

- inline math uses `$...$`;
- display math uses `$$...$$`;
- large systems use MathJax-supported environments such as `aligned`, `array`, `pmatrix`, and `cases`.

So, for example, the notebook should render

$$
x_j(t,v)=a_j+d_jt-P_jv
$$

as a displayed equation, not as raw text.


## Overview

The purpose of this notebook is to give a clean, explainable version of Ritter's Problem I homotopy/decomposition algorithm.

The original paper's notation has many moving pieces. Here we keep the important problem notation

$$
A_j,\qquad B_j,\qquad x_j,\qquad C_j,\qquad c_j,\qquad b_0,\qquad b_j,
$$

but organize the algorithm around one idea:

$$
\boxed{\text{fixed active set} \Rightarrow x_j(t),u_j(t),v(t) \text{ are affine functions of } t.}
$$

Then the only question is: as $t$ decreases, which slack or multiplier hits zero first?


This note rewrites Ritter's algorithm for **Problem I** in a notation that is easier to explain out loud. The goal is to keep the original problem data

$$
A_j,\ B_j,\ x_j,\ c_j,\ C_j,\ b_0,\ b_j

$$

but remove the overly compressed bookkeeping in the paper.

The central idea is:

$$
\text{relax the constraints} \rightarrow \text{start from an easy point} \rightarrow \text{follow the optimal solution as } t \downarrow 0.

$$

For a fixed active set, the solution is affine in $t$. We keep that affine formula until one inactive constraint becomes tight or one active multiplier becomes zero. Then we change the active set and repeat.

---

## 1. Original Problem I


We have $k$ blocks of variables

$$
x_1,\ldots,x_k,
\qquad x_j\in \mathbb{R}^{n_j}.

$$

The objective is block-separable:

$$
\max_{x_1,\ldots,x_k}
\sum_{j=1}^k
\left(c_j^\top x_j-\frac12 x_j^\top C_jx_j\right),

$$

where each $C_j\succ0$. The strict positive definiteness matters because the objective is strictly concave, so KKT conditions are sufficient for global optimality.

The constraints are of two different types.

### Coupling constraints

$$
\sum_{j=1}^k A_jx_j\le b_0.

$$

A coupling row involves several blocks at once. For example, one row might look like

$$
A_{1r}x_1+A_{2r}x_2+\cdots+A_{kr}x_k\le (b_0)_r.

$$

No single block can check this row by itself.

### Local block constraints

$$
B_jx_j\le b_j,
\qquad j=1,\ldots,k.

$$

These are local. The constraint $B_jx_j\le b_j$ only involves block $j$.

So the problem has this structure:

$$
\begin{array}{c|c}
\text{coupling constraints} & \sum_j A_jx_j\le b_0 \\
\text{local block constraints} & B_jx_j\le b_j
\end{array}

$$

---

## 2. Why we do not just put the $B_j$'s into the $A_j$'s


Algebraically, you *can* stack all constraints into one giant matrix. If

$$
x=\begin{pmatrix}x_1\\ \vdots\\ x_k\end{pmatrix},

$$

then the whole feasible region can be written as

$$
\begin{pmatrix}
A_1&A_2&\cdots&A_k\\
B_1&0&\cdots&0\\
0&B_2&\cdots&0\\
\vdots&\vdots&\ddots&\vdots\\
0&0&\cdots&B_k
\end{pmatrix}
\begin{pmatrix}x_1\\x_2\\\vdots\\x_k\end{pmatrix}
\le
\begin{pmatrix}
b_0\\b_1\\b_2\\\vdots\\b_k
\end{pmatrix}.

$$

So the issue is not mathematical possibility. The issue is decomposition.

Ritter separates $A$-constraints from $B$-constraints because they play different algorithmic roles.

### Coupling constraints need shared prices

The multiplier for a coupling row is shared by all blocks. If row $r$ of the coupling constraint is active, it has one multiplier $v_r$, and this same $v_r$ affects every block:

$$
C_jx_j+B_{j,I_j}^\topu_j+A_{j,I_0}^\topv=c_j.

$$

That is why the coupling multiplier $v$ appears in every block subproblem.

### Local constraints should stay local

The multiplier for a local row of $B_jx_j\le b_j$ belongs only to block $j$. It should be part of $u_j$, not part of the global coupling multiplier $v$.

A local constraint from block $j$ cannot be violated or repaired by block $i\ne j$. So there is no reason to put it into the global master system.

### What goes wrong if you put $B$'s into the master

If you treat all $B_j$-rows as if they were coupling rows, then the master multiplier vector becomes huge:

$$
\dim(v)=m_0+\sum_{j=1}^k m_j

$$

instead of just

$$
\dim(v)=m_0.

$$

That destroys the whole point of the method. The method wants the large local parts to be solved separately inside each block, and only the genuinely coupling constraints to appear in the small master system.

### The correct interpretation

The distinction is:

$$
\boxed{
A\text{-rows coordinate the blocks; }B\text{-rows are handled inside the blocks.}
}

$$

You can stack everything into one giant matrix if you want a black-box QP. But then you are no longer exploiting Ritter's decomposition.

---

## 3. Relaxed homotopy problem


We introduce a scalar homotopy parameter $t\ge0$. At $t=0$, we recover the original problem.

The relaxed problem is

$$
\max_{x_1,\ldots,x_k}
\sum_{j=1}^k
\left(c_j^\top x_j-\frac12x_j^\topC_jx_j\right)

$$

subject to

$$
\sum_{j=1}^k A_jx_j\le b_0+e_0t,

$$

and

$$
B_jx_j\le b_j+e_jt,
\qquad j=1,\ldots,k.

$$

The vectors $e_0,e_1,\ldots,e_k$ are relaxation masks. They say which right-hand sides are allowed to move with $t$.

---

## 4. Starting point


First compute the unconstrained maximizer block by block:

$$
x_j^{\mathrm{free}}=C_j^{-1}c_j.

$$

If

$$
\sum_j A_jx_j^{\mathrm{free}}\le b_0,
\qquad
B_jx_j^{\mathrm{free}}\le b_j\quad \forall j,

$$

then stop. The unconstrained maximizer is feasible, so it is the optimal solution.

If not, define the violations

$$
\ell_0=\sum_{j=1}^k A_jx_j^{\mathrm{free}}-b_0,

$$

and

$$
\ell_j=B_jx_j^{\mathrm{free}}-b_j.

$$

A simple relaxation mask is

$$
(e_0)_r=
\begin{cases}
1,&(\ell_0)_r\ge0,\\
0,&(\ell_0)_r<0,
\end{cases}

$$

and

$$
(e_j)_r=
\begin{cases}
1,&(\ell_j)_r\ge0,\\
0,&(\ell_j)_r<0.
\end{cases}

$$

Choose

$$
t_0\ge \max\{(\ell_0)_r,(\ell_j)_r\}.

$$

Then $x^{\mathrm{free}}$ is feasible for the relaxed problem at $t=t_0$.

Now we decrease $t$ from $t_0$ to $0$. The original problem is solved once we reach $t=0$.

---

## 5. Active sets


At a current value of $t$, split the constraints into active and inactive constraints.

### Active coupling set

Let

$$
I_0\subseteq\{1,\ldots,m_0\}

$$

be the set of active coupling rows.

For $r\in I_0$,

$$
\sum_{j=1}^k A_{jr}x_j=(b_0)_r+(e_0)_r t.

$$

Here $A_{jr}$ means row $r$ of $A_j$.

Use row-selection notation:

$$
A_{j,I_0}=\text{rows of }A_j\text{ indexed by }I_0.

$$

Then the active coupling equalities are

$$
\sum_{j=1}^k A_{j,I_0}x_j=b_{0,I_0}+e_{0,I_0}t.

$$

### Active local block sets

For each block $j$, let

$$
I_j\subseteq\{1,\ldots,m_j\}

$$

be the active local rows of $B_jx_j\le b_j+e_jt$.

For $r\in I_j$,

$$
B_{jr}x_j=(b_j)_r+(e_j)_r t.

$$

In row-selection notation,

$$
B_{j,I_j}x_j=b_{j,I_j}+e_{j,I_j}t.

$$

---

## 6. Multipliers


Use:

$$
v(t)=\text{multipliers for active coupling constraints }I_0,

$$

and

$$
u_j(t)=\text{multipliers for active local constraints }I_j.

$$

The full coupling multiplier vector has entries

$$
(v^{\rm full})_{I_0}=v,
\qquad
(v^{\rm full})_{I_0^c}=0.

$$

The full local multiplier vector for block $j$ has entries

$$
(\mu_j)_{I_j}=u_j,
\qquad
(\mu_j)_{I_j^c}=0.

$$

Inactive multipliers are zero because of complementary slackness. If a constraint has positive slack, its multiplier must be zero.

---

## 7. KKT equations for one fixed active set


For the current active sets $(I_0,I_1,\ldots,I_k)$, the active KKT system is:

$$
C_jx_j(t)+B_{j,I_j}^\topu_j(t)+A_{j,I_0}^\topv(t)=c_j,
\qquad j=1,\ldots,k,

$$

$$
B_{j,I_j}x_j(t)=b_{j,I_j}+e_{j,I_j}t,
\qquad j=1,\ldots,k,

$$

and

$$
\sum_{j=1}^k A_{j,I_0}x_j(t)=b_{0,I_0}+e_{0,I_0}t.

$$

These equations enforce stationarity and active equalities only.

They do **not** automatically enforce inactive constraints. Inactive constraints are checked later through slacks.

---

## 8. Solve each block in terms of $v$ and $t$


For block $j$, define the local active KKT matrix

$$
K_j=
\begin{pmatrix}
C_j&B_{j,I_j}^\top\\
B_{j,I_j}&0
\end{pmatrix}.

$$

Assume the rows of $B_{j,I_j}$ are linearly independent. Then $K_j$ is nonsingular.

The block equations are

$$
K_j
\begin{pmatrix}
x_j(t)\\u_j(t)
\end{pmatrix}
=
\begin{pmatrix}
c_j-A_{j,I_0}^\topv(t)\\
b_{j,I_j}+e_{j,I_j}t
\end{pmatrix}.

$$

Because the left matrix is fixed and the right side is affine in $t$ and $v$, the solution is affine in $t$ and $v$.

Write

$$
x_j(t,v)=a_j+d_jt-P_jv,

$$

$$
u_j(t,v)=r_j+s_jt-Q_jv.

$$

The coefficient blocks come from three solves:

$$
K_j
\begin{pmatrix}
a_j\\r_j
\end{pmatrix}
=
\begin{pmatrix}
c_j\\b_{j,I_j}
\end{pmatrix},

$$

$$
K_j
\begin{pmatrix}
d_j\\s_j
\end{pmatrix}
=
\begin{pmatrix}
0\\e_{j,I_j}
\end{pmatrix},

$$

and

$$
K_j
\begin{pmatrix}
P_j\\Q_j
\end{pmatrix}
=
\begin{pmatrix}
A_{j,I_0}^\top\\0
\end{pmatrix}.

$$

So each block has the clean closed form:

$$
\boxed{x_j(t,v)=a_j+d_jt-P_jv.}

$$

and

$$
\boxed{u_j(t,v)=r_j+s_jt-Q_jv.}

$$

If $I_j=\varnothing$, then this reduces to

$$
x_j(t,v)=C_j^{-1}c_j-C_j^{-1}A_{j,I_0}^\topv.

$$

---

## 9. Determine $v$ from active coupling constraints


Now impose the active coupling equalities:

$$
\sum_{j=1}^k A_{j,I_0}x_j=b_{0,I_0}+e_{0,I_0}t.

$$

Substitute

$$
x_j(t,v)=a_j+d_jt-P_jv.

$$

Then

$$
\sum_j A_{j,I_0}(a_j+d_jt-P_jv)=b_{0,I_0}+e_{0,I_0}t.

$$

Collect the $v$-terms:

$$
\left(\sum_j A_{j,I_0}P_j\right)v
=
\left(\sum_j A_{j,I_0}a_j-b_{0,I_0}\right)
+
\left(\sum_j A_{j,I_0}d_j-e_{0,I_0}\right)t.

$$

Define

$$
H=\sum_j A_{j,I_0}P_j,

$$

$$
q^a=\sum_j A_{j,I_0}a_j-b_{0,I_0},

$$

and

$$
q^b=\sum_j A_{j,I_0}d_j-e_{0,I_0}.

$$

Then

$$
Hv=q^a+q^bt.

$$

Assuming $H$ is nonsingular,

$$
\boxed{v(t)=v^a+v^bt}

$$

where

$$
v^a=H^{-1}q^a,
\qquad
v^b=H^{-1}q^b.

$$

If no coupling constraints are active, then $I_0=\varnothing$, there is no master solve, and the full coupling multiplier is simply zero.

---

## 10. Now every unknown is affine in $t$


Substitute $v(t)=v^a+v^bt$ into the block formulas.

For each block,

$$
x_j(t)=a_j+d_jt-P_j(v^a+v^bt).

$$

So

$$
\boxed{x_j(t)=x_j^a+x_j^bt}

$$

where

$$
x_j^a=a_j-P_jv^a,
\qquad
x_j^b=d_j-P_jv^b.

$$

Similarly,

$$
u_j(t)=r_j+s_jt-Q_j(v^a+v^bt),

$$

so

$$
\boxed{u_j(t)=u_j^a+u_j^bt}

$$

where

$$
u_j^a=r_j-Q_jv^a,
\qquad
u_j^b=s_j-Q_jv^b.

$$

And the active coupling multipliers are

$$
\boxed{v(t)=v^a+v^bt.}

$$

This is the whole reason the method works: for a fixed active set, the primal variables and multipliers are straight-line functions of $t$.

---

## 11. What still needs to be checked


The active KKT equations only guarantee:

1. stationarity;
2. active local equalities;
3. active coupling equalities.

To certify that the current active set is valid, we also need:

1. inactive local slacks are nonnegative;
2. inactive coupling slacks are nonnegative;
3. active local multipliers are nonnegative;
4. active coupling multipliers are nonnegative.

That is the full KKT check.

The inactive multipliers are zero by construction. Complementarity is automatic because:

- active constraints have zero slack;
- inactive constraints have zero multiplier.

---

## 12. Generic blocker rule


Every quantity we need to keep nonnegative has the form

$$
g(t)=g^a+g^bt.

$$

At the current parameter value $T$, we know

$$
g(T)\ge0.

$$

We are moving downward in $t$:

$$
T\downarrow 0.

$$

If $g^b\le0$, then decreasing $t$ does not make $g(t)$ smaller. So $g$ cannot block us.

If $g^b>0$, then decreasing $t$ may reduce $g(t)$. The possible blocking value is the root

$$
t_*=-\frac{g^a}{g^b}.

$$

If

$$
0\le t_*<T,

$$

then $t_*$ is a candidate next breakpoint.

Because $t$ moves downward, the first candidate hit is the **largest** candidate root below $T$, not the smallest.

So the next value is

$$
\boxed{
T_{\rm next}=\max\left(\{0\}\cup\{\text{all candidate roots below }T\}\right).
}

$$

This is why Ritter takes a maximum of critical values.

---

## 13. Four blocker types


There are four ways the current active set can stop being valid.

---

### Blocker 1: inactive local constraint becomes tight

Take a local row $r\notin I_j$.

Its slack is

$$
s_{jr}(t)=(b_j)_r+(e_j)_r t-B_{jr}x_j(t).

$$

Using

$$
x_j(t)=x_j^a+x_j^bt,

$$

we get

$$
s_{jr}(t)=s_{jr}^a+s_{jr}^bt

$$

where

$$
s_{jr}^a=(b_j)_r-B_{jr}x_j^a,

$$

and

$$
s_{jr}^b=(e_j)_r-B_{jr}x_j^b.

$$

If $s_{jr}^b>0$, compute

$$
t_{jr}^{\rm add}=-\frac{s_{jr}^a}{s_{jr}^b}.

$$

If this lies in $[0,T)$, it is a candidate root.

At that value,

$$
s_{jr}(t)=0.

$$

So a previously inactive local constraint becomes tight. The update is

$$
I_j\leftarrow I_j\cup\{r\}.

$$

---

### Blocker 2: active local multiplier becomes zero

Take an active local row $r\in I_j$.

Its multiplier is

$$
u_{jr}(t)=(u_j^a)_r+(u_j^b)_r t.

$$

If $(u_j^b)_r>0$, compute

$$
t_{jr}^{\rm drop}=-\frac{(u_j^a)_r}{(u_j^b)_r}.

$$

If this lies in $[0,T)$, it is a candidate root.

At that value,

$$
u_{jr}(t)=0.

$$

If we kept the constraint active below that value, the multiplier would become negative, violating KKT. The update is

$$
I_j\leftarrow I_j\setminus\{r\}.

$$

---

### Blocker 3: inactive coupling constraint becomes tight

Take a coupling row $r\notin I_0$.

Its slack is

$$
s_{0r}(t)=(b_0)_r+(e_0)_r t-
\sum_{j=1}^k A_{jr}x_j(t).

$$

Using

$$
x_j(t)=x_j^a+x_j^bt,

$$

we get

$$
s_{0r}(t)=s_{0r}^a+s_{0r}^bt

$$

where

$$
s_{0r}^a=(b_0)_r-\sum_j A_{jr}x_j^a,

$$

and

$$
s_{0r}^b=(e_0)_r-\sum_j A_{jr}x_j^b.

$$

If $s_{0r}^b>0$, compute

$$
t_{0r}^{\rm add}=-\frac{s_{0r}^a}{s_{0r}^b}.

$$

If this lies in $[0,T)$, it is a candidate root.

At that value, the coupling row becomes tight. The update is

$$
I_0\leftarrow I_0\cup\{r\}.

$$

This changes the master system for $v(t)$.

---

### Blocker 4: active coupling multiplier becomes zero

Take an active coupling row $r\in I_0$.

Its multiplier is

$$
v_r(t)=(v^a)_r+(v^b)_r t.

$$

If $(v^b)_r>0$, compute

$$
t_{0r}^{\rm drop}=-\frac{(v^a)_r}{(v^b)_r}.

$$

If this lies in $[0,T)$, it is a candidate root.

At that value,

$$
v_r(t)=0.

$$

If we kept the coupling row active below that value, its multiplier would become negative. The update is

$$
I_0\leftarrow I_0\setminus\{r\}.

$$

---

## 14. The iteration


At current $T$:

1. Fix active sets $I_0,I_1,\ldots,I_k$.
2. Solve each block system to get

$$
   x_j(t,v)=a_j+d_jt-P_jv,

$$

   and

$$
   u_j(t,v)=r_j+s_jt-Q_jv.

$$

3. Use active coupling equalities to get

$$
   v(t)=v^a+v^bt.

$$

4. Substitute back to get

$$
   x_j(t)=x_j^a+x_j^bt,
   \qquad
   u_j(t)=u_j^a+u_j^bt.

$$

5. Compute all blocker roots.
6. Set

$$
   T_{\rm next}=\max\left(\{0\}\cup\{\text{candidate blocker roots in }[0,T)\}\right).

$$

7. If $T_{\rm next}=0$, stop. The current formula at $t=0$ gives the original optimum.
8. If $T_{\rm next}>0$, update the active set according to the blocker and repeat.

---

## 15. Why the inactive constraints stay valid


They do not stay valid automatically.

They stay valid because we explicitly compute when they would fail.

For every inactive local or coupling row, the slack is affine:

$$
s(t)=s^a+s^bt.

$$

At the current $T$, we have $s(T)\ge0$. If the slack would become negative before $t=0$, it must cross zero first. That zero is one of the candidate blocker roots.

By choosing the largest candidate root below $T$, we stop at the first possible failure as $t$ decreases.

Thus all inactive constraints remain feasible on the interval

$$
[T_{\rm next},T].

$$

---

## 16. Why active multipliers stay valid


Same logic.

Every active multiplier is affine:

$$
u(t)=u^a+u^bt,

$$

or

$$
v(t)=v^a+v^bt.

$$

At the current $T$, all active multipliers are nonnegative. If one would become negative before $t=0$, it must cross zero first. That zero is a blocker root.

By stopping at the largest root below $T$, all active multipliers remain nonnegative on

$$
[T_{\rm next},T].

$$

---

## 17. Why the four cases are exactly enough


The current active set can fail in only two KKT ways:

1. primal feasibility fails;
2. dual feasibility fails.

Primal feasibility fails when an inactive slack becomes negative. The first sign of that is zero slack.

Dual feasibility fails when an active multiplier becomes negative. The first sign of that is zero multiplier.

Each of these can happen either locally or for the coupling constraints. Therefore there are exactly four cases:

$$
\begin{array}{c|c|c}
\text{event} & \text{meaning} & \text{update}\\
\hline
\text{inactive local slack hits zero} & B_j\text{-row becomes tight} & I_j\leftarrow I_j\cup\{r\}\\
\text{active local multiplier hits zero} & B_j\text{-row loses force} & I_j\leftarrow I_j\setminus\{r\}\\
\text{inactive coupling slack hits zero} & A\text{-row becomes tight} & I_0\leftarrow I_0\cup\{r\}\\
\text{active coupling multiplier hits zero} & A\text{-row loses force} & I_0\leftarrow I_0\setminus\{r\}
\end{array}

$$

That is all Ritter's four cases mean.

---

## 18. What happens at a breakpoint


Suppose $T_{\rm next}>0$.

At $t=T_{\rm next}$, one of the four blocker events occurs.

The old affine segment is valid down to this value. At the breakpoint, the old and new descriptions meet at the same point.

### If a constraint is added

The constraint has zero slack at the breakpoint. So adding it as an equality does not change the current point. It only changes the future direction of the path for smaller $t$.

### If a constraint is dropped

The multiplier is zero at the breakpoint. So removing its multiplier from stationarity does not change the current stationarity equation. It only changes the future direction of the path for smaller $t$.

This is why the path is continuous but piecewise linear.

---

## 19. Degeneracy and same-$t$ blockers


The clean story assumes:

1. only one blocker happens at a time;
2. any added active row is linearly independent of the existing active rows.

If several blockers happen at the same $t$, or if a new active row is linearly dependent, then we are in a degenerate case.

### Dropping constraints is safe

If a local multiplier or coupling multiplier becomes zero, dropping that row gives a valid continuation. In Ritter's terminology, the continuation exists for the drop cases.

### Adding constraints needs independence

If a new local or coupling row becomes active, we need the new active row to be linearly independent of the existing active rows. Otherwise, the new KKT matrix or master matrix may not be invertible.

If the added row is dependent, there are two possibilities:

1. there is no feasible continuation below the current $t$;
2. one old active row must be replaced by the new row.

So a practical explanation is:

$$
\boxed{
\text{Nondegenerate add: add the row. Degenerate add: replace an old row or detect infeasibility.}
}

$$

For oral explanation, you can say:

> If a new wall is independent, we add it. If it is dependent, then it is not giving a new direction. Either the feasible set ends there, or the new wall must replace an old wall in the active basis.

---

## 20. Convergence idea


For a fixed active set, the solution and multipliers are affine functions of $t$. The active set is valid on an interval of $t$-values because it is defined by affine inequalities:

$$
\text{inactive slacks}\ge0,
\qquad
\text{active multipliers}\ge0.

$$

The method moves from one interval to the next. At the end of an interval, the active set changes.

There are only finitely many possible active sets:

$$
I_0\subseteq\{1,\ldots,m_0\},
\qquad
I_j\subseteq\{1,\ldots,m_j\}.

$$

In the nondegenerate case, each pivot gives a genuine interval to the left, so $t$ decreases strictly. With the degeneracy rules, same-$t$ cleanup pivots cannot continue forever; they either find a valid active basis or prove infeasibility below that $t$.

Therefore the method terminates with one of two outcomes:

1. $t=0$, in which case the formula $x_j(0)$ gives the optimum of the original problem;
2. a positive $t_p$ below which no feasible continuation exists, in which case the original problem is infeasible.

---

## 21. Why reaching $t=0$ proves optimality


At $t=0$, the relaxed constraints are the original constraints:

$$
\sum_j A_jx_j\le b_0,
\qquad
B_jx_j\le b_j.

$$

The algorithm has maintained the full KKT conditions:

1. stationarity;
2. primal feasibility;
3. dual feasibility;
4. complementarity.

Because the objective is strictly concave and the constraints are linear, these KKT conditions are sufficient for global optimality.

So if the algorithm reaches $t=0$, then

$$
\boxed{x_1(0),\ldots,x_k(0)\text{ solve the original problem.}}

$$

---

## 22. Compact pseudocode


```text
Input: C_j, c_j, A_j, B_j, b_0, b_j.

1. Compute x_j^free = C_j^{-1} c_j.

2. If x^free satisfies all original constraints, stop.

3. Build relaxation masks e_0, e_j and choose t_0 so that x^free is feasible
   for the relaxed constraints at t=t_0.

4. Initialize active sets I_0, I_j from constraints tight at t_0.

5. Set T = t_0.

6. Repeat:

   a. For each block j, form

          K_j = [[C_j, B_{j,I_j}^\top],
                 [B_{j,I_j}, 0]].

   b. Solve block systems to get

          x_j(t,v) = a_j + d_j t - P_j v,
          u_j(t,v) = r_j + s_j t - Q_j v.

   c. Enforce active coupling constraints to get

          v(t) = v^a + v^b t.

   d. Substitute v(t) back into every block:

          x_j(t) = x_j^a + x_j^b t,
          u_j(t) = u_j^a + u_j^b t.

   e. Compute roots for:

          inactive local slacks,
          active local multipliers,
          inactive coupling slacks,
          active coupling multipliers.

   f. Set

          T_next = max({0} union all candidate roots below T).

   g. If T_next = 0:
          return x_j(0).

   h. Otherwise update the active set:

          zero inactive local slack  -> add row to I_j,
          zero active local multiplier -> remove row from I_j,
          zero inactive coupling slack -> add row to I_0,
          zero active coupling multiplier -> remove row from I_0.

   i. Set T = T_next and repeat.
```

---

## 23. How to explain the algorithm in words


Here is a short presentation script.

> We begin from the unconstrained maximizer. If it is infeasible, we relax the right-hand sides by adding $e t$, choosing $t=t_0$ large enough so the unconstrained maximizer is feasible. Then we decrease $t$ to zero.
>
> At any moment, we guess which constraints are active. For that active set, inactive multipliers are zero, and active constraints are equalities. Solving the block KKT systems gives each $x_j$ and local multiplier $u_j$ as affine functions of $t$ and the coupling multiplier $v$.
>
> The active coupling constraints determine $v$ as an affine function of $t$. Once we substitute this back, everything is affine in $t$: the primal variables, the local multipliers, and the coupling multipliers.
>
> The active set remains valid only while inactive slacks stay nonnegative and active multipliers stay nonnegative. Each of those quantities is affine in $t$, so we compute where each one would hit zero. Since $t$ decreases, the next breakpoint is the largest such root below the current $t$.
>
> At the breakpoint, either a slack hits zero, so we add that constraint, or a multiplier hits zero, so we drop that constraint. Then we recompute the affine formula and continue.
>
> If we reach $t=0$, the KKT conditions hold for the original problem, so the point is globally optimal. If the path cannot be continued below some positive $t$, the original feasible set is empty.

---

## 24. The one-line memory version


$$
\boxed{
\text{Fixed active set }\Rightarrow x,u,v\text{ affine in }t;
\quad
\text{first zero slack/multiplier changes the active set.}
}

$$

## Reference

Ritter, K. (1967). *A Decomposition Method for Structured Quadratic Programming Problems*. **Journal of Computer and System Sciences**, 1, 241--260.
